\setcounter{secnumdepth}{0} 

## Proof of Principle (PoP) ##


### Inleiding & Doel
Proof-of-Principle: kan een ML-model mutatieprofielen van longkankercellijnen gebruiken om respons op een drug te voorspellen?

[hier komt uitleg over bewijsstuk]

### Data Inladen ###

In deze stap laden we de originele pickles (mut_matrix.pkl en lung_response_df_pkl) in en worden kopieën gemaakt voor de PoP-analyse. Er wordt een print uitgevoerd van de shape en een klein deel van de data om te zien hoe de ModelID-indexen eruitzien.

## Proof of Principle (PoP) ##


### Inleiding & Doel
Proof-of-Principle: kan een ML-model mutatieprofielen van longkankercellijnen gebruiken om respons op een drug te voorspellen?

[hier komt uitleg over bewijsstuk]

### Data Inladen ###

In deze stap laden we de originele pickles (mut_matrix.pkl en lung_response_df_pkl) in en worden kopieën gemaakt voor de PoP-analyse. Er wordt een print uitgevoerd van de shape en een klein deel van de data om te zien hoe de ModelID-indexen eruitzien.

In [21]:
import pandas as pd

# inladen originele pickles
mut_orig = pd.read_pickle("../data/processed/mut_matrix.pkl")
resp_orig = pd.read_pickle("../data/processed/lung_response_df.pkl")

# maak kopie voor PoP
mut_pop = mut_orig.copy()
response_pop = resp_orig.copy()

print("mut_pop shape:", mut_pop.shape)
print("response_pop shape:", response_pop.shape)

# bekijk eerste 3 rijen en eerste 3 kolommen van mut_pop
print("\nmut_pop sample (3 rows, 3 cols):")
print(mut_pop.iloc[:3, :3])

# bekijk eerste 3 rijen en eerste 3 kolommen van response_pop
print("\nresponse_pop sample (3 rows, 5 cols):")
print(response_pop.iloc[:3, :3])


mut_pop shape: (174, 7358)
response_pop shape: (93, 1451)

mut_pop sample (3 rows, 3 cols):
HugoSymbol  A1CF  A2M  A2ML1
ModelID                     
ACH-000012     0    0      0
ACH-000015     0    0      0
ACH-000021     0    0      0

response_pop sample (3 rows, 5 cols):
       ModelID  8-BROMO-CGMP (BRD:BRD-A00077618-236-07-6)  \
15  ACH-000840                                        NaN   
30  ACH-000921                                        NaN   
89  ACH-000454                                        NaN   

    NORETYNODREL (BRD:BRD-A00758722-001-04-9)  
15                                   0.986123  
30                                   0.913647  
89                                   0.934788  


In [21]:
import pandas as pd

# inladen originele pickles
mut_orig = pd.read_pickle("../data/processed/mut_matrix.pkl")
resp_orig = pd.read_pickle("../data/processed/lung_response_df.pkl")

# maak kopie voor PoP
mut_pop = mut_orig.copy()
response_pop = resp_orig.copy()

print("mut_pop shape:", mut_pop.shape)
print("response_pop shape:", response_pop.shape)

# bekijk eerste 3 rijen en eerste 3 kolommen van mut_pop
print("\nmut_pop sample (3 rows, 3 cols):")
print(mut_pop.iloc[:3, :3])

# bekijk eerste 3 rijen en eerste 3 kolommen van response_pop
print("\nresponse_pop sample (3 rows, 5 cols):")
print(response_pop.iloc[:3, :3])


mut_pop shape: (174, 7358)
response_pop shape: (93, 1451)

mut_pop sample (3 rows, 3 cols):
HugoSymbol  A1CF  A2M  A2ML1
ModelID                     
ACH-000012     0    0      0
ACH-000015     0    0      0
ACH-000021     0    0      0

response_pop sample (3 rows, 5 cols):
       ModelID  8-BROMO-CGMP (BRD:BRD-A00077618-236-07-6)  \
15  ACH-000840                                        NaN   
30  ACH-000921                                        NaN   
89  ACH-000454                                        NaN   

    NORETYNODREL (BRD:BRD-A00758722-001-04-9)  
15                                   0.986123  
30                                   0.913647  
89                                   0.934788  


### Inspectie en fix ModelID

In `response_pop` staat `ModelID` nog als kolom en de index is integer (0, 1, 2, ...).
Dit veroorzaakt later een keyError bij het mergen van de datasets.

Oplossing:

- Zet `ModelID` in `response_pop` om naar de index.  
- Zorg dat de index het juiste format heeft: `ACH-XXXXX`.

In [22]:
# maak ModelID de index

response_pop = response_pop.set_index("ModelID")

# converteer naar ACH-xxxxxx waar nodig
def format_modelid(x):
    if isinstance(x, str) and x.startswith("ACH-"):
        return x
    try:
        return f"ACH-{int(x):06d}"
    except:
        raise ValueError(f"Kan ModelID niet converteren: {x}")

response_pop.index = response_pop.index.map(format_modelid)

# controleer
print("response_pop index after fix (first 3):", response_pop.index[:3])
print("response_pop shape:", response_pop.shape)
print("\nresponse_pop sample (3 rows, 3 cols):")
print(response_pop.iloc[:3, :3])


response_pop index after fix (first 3): Index(['ACH-000840', 'ACH-000921', 'ACH-000454'], dtype='object', name='ModelID')
response_pop shape: (93, 1450)

response_pop sample (3 rows, 3 cols):
            8-BROMO-CGMP (BRD:BRD-A00077618-236-07-6)  \
ModelID                                                 
ACH-000840                                        NaN   
ACH-000921                                        NaN   
ACH-000454                                        NaN   

            NORETYNODREL (BRD:BRD-A00758722-001-04-9)  \
ModelID                                                 
ACH-000840                                   0.986123   
ACH-000921                                   0.913647   
ACH-000454                                   0.934788   

            PREDNISOLONE-ACETATE (BRD:BRD-A01643550-001-04-9)  
ModelID                                                        
ACH-000840                                                NaN  
ACH-000921                                  

### Driver-genen en beschikbare drugs ###

[uitleg over subset driver genen en cellijnen en drug kiezen en over ingebouwde controle die ik heb toegevoegd]

In [23]:
# kies aanwezige longkanker driver genen voor de gekozen drug
all_driver_genes = ["EGFR", "KRAS", "BRAF", "ALK", "ROS1", "ERBB2", "MET", "TP53", "PIK3CA", "STK11"]
pop_genes = [g for g in all_driver_genes if g in mut_pop.columns]
print("Driver genen aanwezig in dataset:", pop_genes)

# selecteer cellijnen met minstens 1 mutatie
gene_subset = mut_pop[pop_genes]
selected_lines = gene_subset.index[gene_subset.sum(axis=1) > 0]

# alleen de cellijnen die ook in response_pop aanwezig zijn
common_lines = selected_lines.intersection(response_pop.index)
print("Aantal cellijnen in subset die ook responsdata hebben:", len(common_lines))

# subset van mutaties en responsdata
gene_subset = mut_pop.loc[common_lines, pop_genes]
resp_subset = response_pop.loc[common_lines]

# zoek drugs zonder missing values in deze subset
complete_drugs_subset = resp_subset.columns[resp_subset.notna().all()].tolist()
# skip eventueel 'ModelID' kolom
selected_drugs = [d for d in complete_drugs_subset if d.lower() != "modelid"]

if len(selected_drugs) == 0:
    raise ValueError("Geen bruikbare drugs zonder missende waarden gevonden in subset.")

# bekijk drugs voor PoP
print(f"Aantal drugs zonder missing values in subset van {len(common_lines)} cellijnen:", len(complete_drugs_subset))
print("\nBeschikbare drugs:")
for drug in complete_drugs_subset:
    print("-", drug)

Driver genen aanwezig in dataset: ['EGFR', 'KRAS', 'BRAF', 'ALK', 'ROS1', 'ERBB2', 'MET', 'TP53', 'PIK3CA', 'STK11']
Aantal cellijnen in subset die ook responsdata hebben: 89
Aantal drugs zonder missing values in subset van 89 cellijnen: 7

Beschikbare drugs:
- BERZOSERTIB (BRD:BRD-K04701033-001-03-9)
- PF-05212384 (BRD:BRD-K07955840-001-02-3)
- VERUBULIN (BRD:BRD-K42673188-001-01-1)
- OTS167 (BRD:BRD-K53417444-003-03-1)
- RUBITECAN (BRD:BRD-K79821389-001-03-5)
- TOSEDOSTAT (BRD:BRD-K92241597-001-06-0)
- SB-2343 (BRD:BRD-K98795921-001-01-7)


### Subset van cellijnen en features maken

Voor deze PoP heb ik de volgende keuzes gemaakt:

- Drug: PF-05212384 (BRD:BRD-K07955840-001-02-3).  
    Dit is een PI3K/mTOR inhibitor, relevant voor mutaties in PIK3CA, één van de driver-genen in onze subset. De responsdata voor deze drug is volledig aanwezig in de geselecteerde cellijnen.
- Samples (cellijnen): 30 uit de 89 beschikbare cellijnen.  
    Dit is een kleine subset om snel een proof-of-principle model te trainen en evalueren.
- Features (mutaties):
    - 10 driver-genen: top 10 meest biologisch relevante genen. Onder deze genen is PIK3CA waarvan de mutaties naar verwachting invloed hebben op de repons op de gekozen drug PF-05212384 (PI3K/mTOR inhibitor).
    - 20 random gekozen genen.
- Train/test split: 80/20
  Voldoende cellijnen om een klein model te trainen (24 cellijnen) en testen (6 cellijnen) voor evaluatie.



In [35]:
import numpy as np
import numpy as np

# gelabelde output/drug
label_pop = "PF-05212384 (BRD:BRD-K07955840-001-02-3)"

# response_pop dezelfde index als mut_pop
if response_pop.index.name != 'ModelID':
    response_pop.set_index('ModelID', inplace=True)

# subset van 30 random cellijnen (alleen als ze in beide dataframes voorkomen)
np.random.seed(42)
common_subset = list(set(mut_pop.index).intersection(response_pop.index))
subset_lines_pop = np.random.choice(common_subset, size=30, replace=False)

# subset van features (top 10 driver-genen + 20 random genen)
non_driver_genes = [g for g in mut_pop.columns if g not in pop_genes]
random_non_driver = list(np.random.choice(non_driver_genes, size=20, replace=False))
subset_genes_pop = pop_genes + random_non_driver

# subset van mutaties en respons
X_pop = mut_pop.loc[subset_lines_pop, subset_genes_pop]
y_pop = response_pop.loc[subset_lines_pop, label_pop]

print("X_pop shape:", X_pop.shape)
print("y_pop shape:", y_pop.shape)
print("\nVoorbeeld X_pop (eerste 3 rijen, eerste 5 kolommen):")
print(X_pop.iloc[:3, :5])
print("\nVoorbeeld y_pop (eerste 3 rijen):")
print(y_pop.head(3))


X_pop shape: (30, 30)
y_pop shape: (30,)

Voorbeeld X_pop (eerste 3 rijen, eerste 5 kolommen):
HugoSymbol  EGFR  KRAS  BRAF  ALK  ROS1
ModelID                                
ACH-000062     0     0     0    0     0
ACH-000627     0     0     0    0     0
ACH-000444     0     1     0    0     0

Voorbeeld y_pop (eerste 3 rijen):
ModelID
ACH-000062    0.602987
ACH-000627    0.600329
ACH-000444    0.638270
Name: PF-05212384 (BRD:BRD-K07955840-001-02-3), dtype: float64


### Train/test split en scikit-learn model ###

[uitleg over train/test split (en model?)]

In [36]:
from sklearn.model_selection import train_test_split

# train/test split (80/20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_pop,
    y_pop,
    test_size=0.2, # bij 30 samples dus 6 test, 24 train)
    random_state=42 # reproduceerbare split?
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (24, 30)
X_test shape: (6, 30)
y_train shape: (24,)
y_test shape: (6,)


### RandomForestRegressor trainen + voorspellen + evalueren



#### Evalueren ####
 
Voor het evalueren van het Random Forest regressiemodel heb ik verschillende evaluatiemaatstaven gebruikt.

- **R^2** (R-squared, determinatiecoëfficiënt)
R^2 is een maat voor hoe goed het model de variatie in de respons (AUC) kan verklaren op basis van mutaties.

Interpretatie:  
-R^2 = 1 -> perfecte voorspelling.

-R^2 = 0 -> model voorspelt  even goed als het gemiddelde, geen voorspellend vermogen.

-R^2 < 0 -> model presteert slechter dan gemiddeld.

R^2 zegt niets over hoeveel voorspellingen exact correct zijn, maar geeft aan hoeveel van de verschillen in AUC tussen samples door het model kan worden verklaard. 

- **RMSE** (Root Mean Squared Error)
RSME geeft de gemiddelde fout van het model weer in dezelfde eenheid als de respons (AUC)

Gemiddeld wijkt de voorspelling van het model ±RMSE af van de werkelijke AUC.

- **(optioneel) MAE** (Mean Absolute Error) 
MAE is het gemiddelde van de absolute fouten tussen voorspelde en werkelijke AUC.

Het voordeel is dat MAE niet sterk wordt beinvloed door een paar extreem afwijkende waarden in de data. Het laat dus een realistischer beeld zien van de typische fout van het model.

Door R^2, RMSE en MAE te combineren ontstaat er een compleet beeld van zowel het voorspellend vermogen van het model (R^2) als de grootte van de voorspelfouten in de schaal van de AUC (RMSE en MAE). Voor deze proof of principle is vooral relevant dat kan worden aangetoond dat het model enig signaal uit de mutaties kan oppikken en dat de volledige machine-learning pipeline succesvol kan worden uitgevoerd.

In [37]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

# model aanmaken
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# trainen (fitten)
model.fit(X_train, y_train)

# voorspellen op testset
y_pred = model.predict(X_test)

# evalueren
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

print("Resultaten:")
print("R² score:", r2)
print("MSE:", mse)
print("RMSE:", rmse)
print("MAE:", mae)

print("y_min:", y_pop.min())
print("y_max:", y_pop.max())

Resultaten:
R² score: 0.08349045280315626
MSE: 0.0035071847026042122
RMSE: 0.05922148852067307
MAE: 0.04720569490411688
y_min: 0.529217848063749
y_max: 0.777116758208797


### Conclusie ###

De R^2 score betekend dat ongeveer 8% van de variatie in de gemeten AUC-waarden kan worden verklaard door het model. Het model pikt dus een klein signaal uit de mutaties. Dit is normaal bij slechts 30 samples en 30 binaire featuers. Voor een proof-of-principle is dit voldoende bewijs dat de pipeline werkt en dat er ten minste een zwak patroon in de data aanwezig is.

De RMSE geeft aan dat de voorspellingen gemiddeld ongeveer 0.06 AUC-punten van de gemeten responswaarden afwijken. De MAE laat zien dat de gemiddelde afwijking 0.047 AUC bedraagt en ligt zoals verwacht iets lager dan de RMSE, omdat RMSE grotere fouten zwaarder meeweegt.

Of deze fouten groot of klein zijn, hangt af van de spreding van de responswaarden in de dataset. In deze subset liggen de AUC-responswaarden afgerond tussen de 0.53 en 0.78. Relatief gezien betekent dat een fout van ongeveer 0.05 dat de voorspellingen een klein tot matig deel van de totale variatie missen, wat logisch is bij een kleine dataset. Fouten worden het beste beoordeeld in verhouding tot de spreiding in de responswaarden, en niet aan een vaste grens.

De volledige pipeline werkt (inladen > train/test split > trainen > voorspellen > evalueren). De evaluatiematen zijn berekend en geinterpreteerd. De resultaten laten zien dat er enige voorspellende waarde in de mutaties zit, en de cijfers zijn realistisch gezien de kleine dataset. De proof-of-principle is geslaagd.

### Feature importance ###

[Hier ben ik nog mee bezig, komt later]

Bron in scikit-learn:
User Guide → Supervised Learning → Regression
Ensemble methods → RandomForestRegressor
Metrics → Regression metrics
